# Task 1 - 
## Valuing the following four trades (OTC option positions):

*all valuations are based on close of market Friday 16 May 2025 

**1. European call option on BHP**
- exp 15 Sep 2027
- strike price: 98% BHP closing price on 16 May 2025

**2. American put option on CBA**
- exp 15 Sep 2027
- strike price: $170 (fixed)

**3. European up-and-in barrier call option on WES**
- exp 15 Sep 2027
- strike price: $80
- all in barrier level: $100

**4. European basket call option**
- weightings: [10% BHP, 35% CSL, 15% WDS, 40% MQG]
- exp 17 July 2027
- strike price: $175
        


### Defining Classes and Objects 

In [ ]:
# import libraries 
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from datetime import datetime, timedelta
import requests
import textblob as tb
import lxml as lx

In [ ]:
# defining classes and objects

# define option class
class Option:
    def __init__(self, underlying, strike, expiry, option_type, style):
        self.underlying = underlying        # e.g. 'BHP' or a basket
        self.strike = strike                # float
        self.spot_price = spot_price        # <- this is the price as of 16 May 2025               
        self.expiry = expiry                # datetime.date
        self.option_type = option_type      # 'call' or 'put'
        self.style = style                  # 'European', 'American', etc.

    def value(self, volatility, risk_free_rate):
    # Use self.spot_price directly in Black-Scholes or binomial tree
        # Pricing logic here
        pass


# define subclasses for each option type 
class EuropeanOption(Option):
    def __init__(self, underlying, spot_price, strike, expiry, option_type):
        super().__init__(underlying, spot_price, strike, expiry, option_type, style='European')

    def value(self, volatility, risk_free_rate):
        # You can plug in Black-Scholes here
        pass


class AmericanOption(Option):
    def __init__(self, underlying, spot_price, strike, expiry, option_type):
        super().__init__(underlying, spot_price, strike, expiry, option_type, style='American')

    def value(self, volatility, risk_free_rate):
        # Placeholder for binomial tree or other early exercise logic
        pass


class BarrierOption(Option):
    def __init__(self, underlying, spot_price, strike, expiry, option_type, barrier_level, barrier_type):
        super().__init__(underlying, spot_price, strike, expiry, option_type, style='European')
        self.barrier_level = barrier_level        # e.g. 100.0
        self.barrier_type = barrier_type          # e.g. 'up-and-in', 'down-and-out'

    def value(self, volatility, risk_free_rate):
        # Pricing logic for barrier options
        pass


class BasketOption(Option):
    def __init__(self, basket, spot_prices, weights, strike, expiry, option_type):
        # Compute the basket spot price as weighted sum of individual prices
        basket_price = sum(spot_prices[ticker] * weights[ticker] for ticker in basket)

        # Store underlying as basket (list of tickers)
        super().__init__(underlying=basket, spot_price=basket_price, strike=strike, expiry=expiry, option_type=option_type, style='European')

        self.weights = weights                  # dict: {'BHP.AX': 0.1, 'CSL.AX': 0.35, ...}
        self.spot_prices = spot_prices          # dict: {'BHP.AX': 45.67, ...}

    def value(self, volatility, risk_free_rate):
        # Monte Carlo or analytical approximation for basket options
        pass


# define portfolio class
class OptionPortfolio:
    def __init__(self):
        self.positions = []

    def add_option(self, option, quantity):
        self.positions.append((option, quantity))

    def total_value(self, market_data):
        total = 0.0
        for option, quantity in self.positions:
            total += quantity * option.value(market_data)
        return total


In [ ]:
import yfinance as yf
import datetime

# Download adjusted closing prices for the valuation date
tickers = ['BHP.AX', 'CBA.AX', 'WES.AX', 'CSL.AX', 'WDS.AX', 'MQG.AX']
date = "2025-05-16"

# Download price data
data = yf.download(tickers, start=date, end="2025-05-17")['Adj Close']
closing_prices = data.loc[date].to_dict()

# --- Option Classes  ---
# - EuropeanOption
# - AmericanOption
# - BarrierOption
# - BasketOption

# --- 1. European Call Option on BHP (strike = 98% of spot) ---
bhp_price = closing_prices['BHP.AX']
bhp_strike = 0.98 * bhp_price
bhp_option = EuropeanOption(
    underlying='BHP.AX',
    spot_price=bhp_price,
    strike=bhp_strike,
    expiry='2027-09-15',
    option_type='call'
)

# --- 2. American Put Option on CBA (strike = $170.00) ---
cba_price = closing_prices['CBA.AX']
cba_option = AmericanOption(
    underlying='CBA.AX',
    spot_price=cba_price,
    strike=170.00,
    expiry='2026-05-15',
    option_type='put'
)

# --- 3. European Up-and-In Barrier Call Option on WES ---
wes_price = closing_prices['WES.AX']
wes_option = BarrierOption(
    underlying='WES.AX',
    spot_price=wes_price,
    strike=80.00,
    expiry='2027-09-15',
    option_type='call',
    barrier_level=100.00,
    barrier_type='up-and-in'
)

# --- 4. European Basket Call Option (strike = $175.00) ---
basket_weights = {
    'BHP.AX': 0.10,
    'CSL.AX': 0.35,
    'WDS.AX': 0.15,
    'MQG.AX': 0.40
}

# Extract relevant spot prices from closing_prices
basket_spot_prices = {ticker: closing_prices[ticker] for ticker in basket_weights.keys()}

basket_option = BasketOption(
    basket=list(basket_weights.keys()),
    spot_prices=basket_spot_prices,
    weights=basket_weights,
    strike=175.00,
    expiry='2025-07-17',
    option_type='call'
)



# Task 2 - 
## Hedging Strategies and information for the portfolio containing all four positions

